# Task 4 – Distributed Computing
**Focus:** Spark UI, Caching, Persisting, Resource Configuration


In [1]:
# ============================================================
# TASK 4: Distributed Computing Optimisation
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.storagelevel import StorageLevel
import time

# Resource-configured SparkSession
spark = SparkSession.builder \
    .appName('7006SCN_Task4_DistributedComputing') \
    .config('spark.executor.memory', '4g') \
    .config('spark.executor.cores', '2') \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '200') \
    .config('spark.default.parallelism', '8') \
    .config('spark.memory.fraction', '0.8') \
    .config('spark.memory.storageFraction', '0.3') \
    .getOrCreate()

print('SparkSession with resource configuration initialised')
print('\nResource Configuration:')
print(f'  executor.memory: 4g')
print(f'  executor.cores: 2')
print(f'  driver.memory: 4g')
print(f'  shuffle.partitions: 200')
print(f'  default.parallelism: 8')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 14:19:06 WARN Utils: Your hostname, Naveenrajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.34 instead (on interface en0)
26/06/29 14:19:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 14:19:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 14:19:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/29 14:19:07 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/29 14:19:07 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/29 14:19:0

SparkSession with resource configuration initialised

Resource Configuration:
  executor.memory: 4g
  executor.cores: 2
  driver.memory: 4g
  shuffle.partitions: 200
  default.parallelism: 8


In [2]:
# ============================================================
# LOAD DATA & REPARTITION STRATEGY
# ============================================================

df = spark.read.parquet('./data/train_processed.parquet')
print(f'Default partitions after load: {df.rdd.getNumPartitions()}')

# Repartition to optimise parallelism
# Using 8 partitions to match default.parallelism
df_repartitioned = df.repartition(8)
print(f'Partitions after repartition(8): {df_repartitioned.rdd.getNumPartitions()}')

# CACHE the training DataFrame (used repeatedly across models)
# MEMORY_AND_DISK: Stores in memory, spills to disk if needed
df_repartitioned.persist(StorageLevel.MEMORY_AND_DISK)
print('DataFrame cached with MEMORY_AND_DISK storage level')
print('Justification: Training data is accessed multiple times during CrossValidation')
print('MEMORY_AND_DISK prevents recomputation while tolerating memory pressure')

# Trigger caching action
start = time.time()
count = df_repartitioned.count()
print(f'\nFirst count (materialises cache): {count:,} rows in {time.time()-start:.2f}s')

start = time.time()
count2 = df_repartitioned.count()
print(f'Second count (from cache): {count2:,} rows in {time.time()-start:.2f}s')

Default partitions after load: 8


[Stage 1:=======>                                                   (1 + 7) / 8]

Partitions after repartition(8): 8
DataFrame cached with MEMORY_AND_DISK storage level
Justification: Training data is accessed multiple times during CrossValidation
MEMORY_AND_DISK prevents recomputation while tolerating memory pressure


[Stage 4:>                                                          (0 + 8) / 8]


First count (materialises cache): 2,297,635 rows in 1.65s
Second count (from cache): 2,297,635 rows in 0.11s


In [3]:
# ============================================================
# SPARK UI EVIDENCE
# Access Spark UI at: http://localhost:4040
# Take screenshot of:
#   - Jobs tab (completed jobs)
#   - Stages tab (task distribution)
#   - Storage tab (cached RDDs)
# ============================================================

# Trigger multiple transformations to generate Spark UI evidence
from pyspark.sql.functions import avg, stddev

# Job 1: Aggregation
print('Running aggregation job (generates Spark UI evidence)...')
start = time.time()
stats = df_repartitioned.agg(
    avg('trip_distance').alias('avg_distance'),
    stddev('fare_amount').alias('std_fare')
).collect()
print(f'Aggregation completed in {time.time()-start:.2f}s')
print(f'Results: {stats}')

# Job 2: Filter + count (shows task skew if any)
print('\nRunning filter job...')
start = time.time()
high_tip_count = df_repartitioned.filter(col('high_tip') == 1).count()
print(f'High tip records: {high_tip_count:,} in {time.time()-start:.2f}s')

print('\n>>> SCREENSHOT THE SPARK UI NOW at http://localhost:4040 <<<')
print('Capture: Jobs page, Stages page, Storage page')

Running aggregation job (generates Spark UI evidence)...
Aggregation completed in 0.23s
Results: [Row(avg_distance=3.4302825383492093, std_fare=16.913618179203258)]

Running filter job...
High tip records: 1,407,319 in 0.18s

>>> SCREENSHOT THE SPARK UI NOW at http://localhost:4040 <<<
Capture: Jobs page, Stages page, Storage page


In [4]:
# ============================================================
# CACHING vs PERSISTING COMPARISON
# ============================================================

test_df = spark.read.parquet('./data/test_processed.parquet')

# cache() = MEMORY_ONLY
# Suitable for: smaller DataFrames that fit in memory
test_df.cache()
test_df.count()  # materialise
print('Test DataFrame cached with MEMORY_ONLY (cache())')
print('Justification: Test set is smaller, used only for final evaluation')

# persist(MEMORY_AND_DISK) = Training DataFrame
print('\nTrain DataFrame persisted with MEMORY_AND_DISK (persist())')
print('Justification: Large training set may not fit in memory during CrossValidation')

print('\nStorage levels used:')
print('  MEMORY_ONLY:       Fast access, may recompute on eviction')
print('  MEMORY_AND_DISK:   Slower but reliable, no recomputation')
print('  DISK_ONLY:         Slowest, used if memory is critically low')

Test DataFrame cached with MEMORY_ONLY (cache())
Justification: Test set is smaller, used only for final evaluation

Train DataFrame persisted with MEMORY_AND_DISK (persist())
Justification: Large training set may not fit in memory during CrossValidation

Storage levels used:
  MEMORY_ONLY:       Fast access, may recompute on eviction
  MEMORY_AND_DISK:   Slower but reliable, no recomputation
  DISK_ONLY:         Slowest, used if memory is critically low


In [5]:
# ============================================================
# RESOURCE CONFIGURATION SCREENSHOT
# ============================================================

print('=== Spark Configuration ===')
configs_to_show = [
    'spark.executor.memory',
    'spark.executor.cores',
    'spark.driver.memory',
    'spark.sql.shuffle.partitions',
    'spark.default.parallelism',
    'spark.memory.fraction',
    'spark.memory.storageFraction'
]
for c in configs_to_show:
    val = spark.conf.get(c, 'not set')
    print(f'  {c}: {val}')

print('\n=== Justification of Resource Choices ===')
print('executor.memory=4g: Sufficient to hold feature vectors for 30M+ row dataset')
print('executor.cores=2: Balances parallelism without overwhelming single machine')
print('shuffle.partitions=200: Default Spark value, suitable for large shuffle operations')
print('memory.fraction=0.8: Maximises usable memory for execution and storage')
print('default.parallelism=8: Matches available CPU cores for optimal task distribution')

# Clean up
df_repartitioned.unpersist()
test_df.unpersist()
print('\nDataFrames unpersisted')

# spark.stop()

=== Spark Configuration ===
  spark.executor.memory: 4g
  spark.executor.cores: 2
  spark.driver.memory: 4g
  spark.sql.shuffle.partitions: 200
  spark.default.parallelism: 8
  spark.memory.fraction: 0.8
  spark.memory.storageFraction: 0.3

=== Justification of Resource Choices ===
executor.memory=4g: Sufficient to hold feature vectors for 30M+ row dataset
executor.cores=2: Balances parallelism without overwhelming single machine
shuffle.partitions=200: Default Spark value, suitable for large shuffle operations
memory.fraction=0.8: Maximises usable memory for execution and storage
default.parallelism=8: Matches available CPU cores for optimal task distribution

DataFrames unpersisted


26/06/29 15:26:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 168006 ms exceeds timeout 120000 ms
26/06/29 15:26:54 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/29 15:26:55 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o